# Lecture 26 Worksheet: Convolution, Pooling and a First CNN
**NAVTTC / NIAI AI/ML Trainer Cohort** — works in Google Colab. Fill every `___` blank and run top to bottom; check lines print `OK`. Tasks 1 to 4 use numpy only. Tasks 5 and 6 use TensorFlow, which Colab already has.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)

# The 6x6 'L' image from the lecture (slide 6): 1 = ink
img = np.zeros((6, 6), dtype=int)
img[0:5, 1] = 1          # the vertical bar
img[4, 1:5] = 1          # the horizontal bar
# A filter that likes a thin vertical line
V = np.array([[-1, 2, -1],
              [-1, 2, -1],
              [-1, 2, -1]])
print(img)

## Task 1: Slide the filter over the image
For every position, take the 3 × 3 patch, multiply it by the filter **element by element**, and add up the 9 products. Fill the blanks.

In [ ]:
def conv2d_valid(image, filt):
    f = filt.shape[0]
    n = image.shape[0] - f + ___            # output size, no padding, stride 1
    out = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            patch = image[i:i+f, j:j+f]
            out[i, j] = np.sum(patch * ___)   # multiply, then add
    return out

fmap = conv2d_valid(img, V)
print(fmap)
expected = np.array([[6, -3, 0, 0], [6, -3, 0, 0], [5, -2, 0, 1], [3, -1, 0, 1]])
assert (fmap == expected).all(); print("OK")

In [ ]:
# Turn the filter on its side to get a horizontal-line detector, and look at both maps
H = V.T
fmap_h = conv2d_valid(img, H)
print(fmap_h)
fig, ax = plt.subplots(1, 3, figsize=(10, 3))
for a, m, t in zip(ax, [img, fmap, fmap_h], ["image", "vertical filter", "horizontal filter"]):
    a.imshow(m, cmap="RdBu_r"); a.set_title(t); a.axis("off")
plt.show()
assert fmap_h[3].tolist() == [3, 5, 6, 4]; print("OK")

## Task 2: ReLU and max pooling
ReLU keeps the positive numbers and turns the negatives into 0. Max pooling keeps the biggest number in each 2 × 2 block.

In [ ]:
def relu(a):
    return np.maximum(___, a)

def max_pool2(a):
    n = a.shape[0] // 2
    out = np.zeros((n, n), dtype=a.dtype)
    for i in range(n):
        for j in range(n):
            out[i, j] = a[2*i:2*i+2, 2*j:2*j+2].___()   # the biggest number in the block
    return out

r = relu(fmap)
p = max_pool2(r)
print(r); print(p)
assert p.tolist() == [[6, 0], [5, 1]]; print("OK")

## Task 3: The output-size formula
`(n - f + 2p) / s + 1`, rounded down. Write it as a function and test it on the table from slide 8.

In [ ]:
def out_size(n, f, p, s):
    return (n - f + ___*p) // s + 1

tests = [((6, 3, 0, 1), 4), ((6, 3, 1, 1), 6), ((6, 3, 0, 2), 2),
         ((8, 3, 0, 1), 6), ((7, 3, 0, 2), 3), ((28, 5, 0, 1), 24)]
for args, want in tests:
    assert out_size(*args) == want, (args, out_size(*args), want)
print("OK")

## Task 4: Count the parameters, then check with Keras
Rule: `(f * f * channels_in + 1) * number_of_filters`. The `+ 1` is each filter's bias. Then build the small CNN from the lecture (slide 13) and compare your count with `model.count_params()`.

In [ ]:
import os; os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import tensorflow as tf
L = tf.keras.layers

def conv_params(f, c_in, k):
    return (f * f * ___ + 1) * k

assert conv_params(3, 1, 8) == 80
assert conv_params(3, 3, 8) == 224
assert conv_params(3, 8, 16) == 1168
print("conv layer counts OK")

def make_cnn():
    return tf.keras.Sequential([
        L.Input((8, 8, 1)),
        L.Conv2D(___, 3, activation="relu"),   # 8 filters of 3x3
        L.MaxPooling2D(___),                   # halve height and width
        L.Flatten(),
        L.Dense(10)])                          # raw scores

cnn = make_cnn(); cnn.summary()
by_hand = conv_params(3, 1, 8) + (3*3*8) * 10 + 10      # conv + dense (72 inputs -> 10)
print("by hand:", by_hand, " Keras:", cnn.count_params())
assert by_hand == cnn.count_params() == 810; print("OK")

## Task 5: Dense network or CNN on the digits?
scikit-learn has 1797 small handwritten digits (8 × 8, values 0 to 16). Train the dense network and the CNN on the same 1000 digits and compare their accuracy on 500 new digits.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

d = load_digits()
X = (d.images / ___).astype("float32")      # scale 0-16 to 0-1
y = d.target
Xtr, Xte, ytr, yte = train_test_split(X, y, train_size=1000, test_size=500, random_state=0, stratify=y)
Xtr, Xte = Xtr[..., None], Xte[..., None]    # add the channel axis: (n, 8, 8, 1)
print(Xtr.shape, Xte.shape)

def make_dense():
    return tf.keras.Sequential([L.Input((8, 8, 1)), L.Flatten(),
                                L.Dense(32, activation="relu"), L.Dense(10)])

def train_eval(make, Xtr, ytr, Xte, yte, epochs=30, seed=0):
    tf.keras.utils.set_random_seed(seed)
    m = make()
    m.compile("adam", tf.keras.losses.SparseCategoricalCrossentropy(from_logits=___), metrics=["accuracy"])
    m.fit(Xtr, ytr, epochs=epochs, batch_size=32, verbose=0)
    return m.evaluate(Xte, yte, verbose=0)[1], m.count_params()

acc_d, n_d = train_eval(make_dense, Xtr, ytr, Xte, yte)
acc_c, n_c = train_eval(make_cnn, Xtr, ytr, Xte, yte)
print(f"dense: {acc_d:.3f}  ({n_d} parameters)")
print(f"CNN:   {acc_c:.3f}  ({n_c} parameters)")
assert acc_d > 0.85 and acc_c > 0.85; print("OK")

**Your answer** (one or two sentences): which model was more accurate here, and which used fewer parameters? Run it with `random_state=1` and `2` as well. Did the winner change?

_Write here:_

## Task 6: Now move the digit around
Enlarge each digit to 16 × 16 and paste it on a 24 × 24 canvas, either in the centre or at a random position. A dense network has to relearn every position; a CNN can reuse its filters.

In [ ]:
def canvas(imgs, random_pos, seed=0):
    Z = np.kron(imgs[..., 0], np.ones((2, 2), dtype="float32"))     # 8x8 -> 16x16
    rng = np.random.default_rng(seed)
    out = np.zeros((len(Z), 24, 24), dtype="float32")
    for i, z in enumerate(Z):
        r, c = rng.integers(0, 9, 2) if random_pos else (4, 4)
        out[i, r:r+16, c:c+16] = z
    return out[..., None]

def make_dense24():
    return tf.keras.Sequential([L.Input((24, 24, 1)), L.Flatten(),
                                L.Dense(32, activation="relu"), L.Dense(10)])
def make_cnn24():
    return tf.keras.Sequential([L.Input((24, 24, 1)),
        L.Conv2D(8, 3, activation="relu"), L.MaxPooling2D(2),
        L.Conv2D(16, 3, activation="relu"), L.MaxPooling2D(2),
        L.Flatten(), L.Dense(10)])

res = {}
for random_pos in (False, True):
    A, B = canvas(Xtr, random_pos, seed=1), canvas(Xte, random_pos, seed=2)
    for name, mk in (("dense", make_dense24), ("cnn", make_cnn24)):
        acc, n = train_eval(mk, A, ytr, B, yte)
        res[(name, random_pos)] = acc
        print(f"{'random' if random_pos else 'centred':8s} {name:6s} {acc:.3f}  ({n} parameters)")
assert min(res.values()) > 0.4; print("OK")

**Your answer:** how much did each network lose when the digit moved? Which lost less, and why do you think so (use the words *shared weights* and *pooling*)?

_Write here:_

## Task 7: Your own project
Write 3 to 4 sentences. Does the data in your project have spatial structure (images, audio spectrograms, sensor grids), or is it a table of separate columns? Would you try a CNN or a simpler model first, and what would be your baseline?

_Write here:_